# Limpieza y transformación del catálogo de Netflix

Este notebook toma el archivo `netflix1.csv` y lo deja listo para análisis o para cargarlo en una herramienta de visualización (Power BI, Tableau, etc.). Cada celda explica qué hace y por qué.

## Importar librerías y cargar los datos

In [ ]:
import pandas as pd

# Cargamos el CSV en un DataFrame. Todas las columnas llegan como texto (object)
# excepto 'release_year', que pandas detecta automáticamente como número entero.

ruta_dataset = r"data/netflix1.csv 
# Vistazo rápido: cuántas filas/columnas tenemos y cómo se ven los primeros registros
print(df.shape)
df.head()

## Revisar la estructura del dataset

Antes de limpiar, conviene confirmar el tipo de dato de cada columna y detectar valores faltantes o incompletos (por ejemplo, texto como "Not Given" en vez de un NaN real).

In [ ]:
# Tipos de dato por columna

df.info()

In [ ]:
# pandas no marca nulos porque el dataset usa el texto 'Not Given' como relleno
# en vez de dejar la celda vacía. Por eso .isnull().sum() da 0 en todas las columnas,
# aunque sí falte información real.
print('Nulos detectados por pandas:')
print(df.isnull().sum())

print(f'\nHay { (df["director"] == "Not Given").sum()} filas con director no especificado, es decir el {len(df[df["director"] == "Not Given"]) / len(df) * 100:.2f}% ')
print(f'Hay { (df["country"] == "Not Given").sum() } filas con país no especificado, es decir el {len(df[df["country"] == "Not Given"]) / len(df) * 100:.2f}%')

## Eliminar duplicados

Buscamos filas que compartan título, tipo, año de estreno y país: si coinciden en las cuatro, es casi seguro que son el mismo título cargado dos veces.

In [ ]:
# keep='first' conserva la primera aparición y marca las repeticiones siguientes como duplicado
duplicados = df.duplicated(subset=['title', 'type', 'release_year', 'country'], keep='first')
print('Duplicados encontrados:', duplicados.sum())

# Nos quedamos solo con las filas que NO son duplicado (~ invierte el booleano)
df = df[~duplicados].copy()
print('Filas después de limpiar duplicados:', len(df))

## Convertir la fecha a formato de fecha real

`date_added` llega como texto ('9/25/2021'). Convertirla a `datetime` permite filtrar por año/mes y hacer análisis de series de tiempo más adelante.

In [ ]:
# El formato original es mes/día/año (m/d/Y), típico del inglés estadounidense
df['date_added'] = pd.to_datetime(df['date_added'], format='%m/%d/%Y')

# Comprobamos que el tipo de dato cambió correctamente
print(df['date_added'].dtype)
df['date_added'].head()

## Eliminar una columna 

Ahora vamos a eliminar una columna que no nos está aportando mucha información relevante,
para esto elegimos la columna "show_id" ya que es un identificador único para cada película o serie y no nos aporta información 

In [ ]:
# Para eliminar múltiples columnas se usa df.drop(columns=['columna1', 'columna2'])
df = df.drop(columns=['show_id'])

df.head()

## Quedarnos con el país y el género principal

Muchas filas tienen varios países o géneros separados por comas (ej. `"France, United States"`). Para gráficos simples (top países, top géneros) conviene tener una columna con un solo valor representativo: el primero de la lista.

In [ ]:
# Crear la columna con el primer país limpio
df['main_country'] = df['country'].str.split(',').str[0].str.strip()
# Crear la columna con el primer género limpio
df['main_genre'] = df['listed_in'].str.split(',').str[0].str.strip()

#además vamos a mostrar los 5 países y géneros más frecuentes en el dataset 
print(df['main_country'].value_counts().sort_values(ascending=False).head())
print()
print(df['main_genre'].value_counts().sort_values(ascending=False).head())

## Guardar el dataset limpio

Exportamos un nuevo CSV para no modificar el archivo original y poder usar esta versión limpia directamente en Power BI u otra herramienta.

In [ ]:
df.to_csv('netflix1_limpio.csv', index=False, encoding='utf-8')
print('Archivo guardado: netflix1_limpio.csv')
print('Filas finales:', len(df), '| Columnas finales:', len(df.columns))